## **RDD Transformations and Actions**

### **Spark 설정하기**

In [1]:
# [+] SparkConf, SparkContext 임포트
from pyspark import SparkConf, SparkContext

In [2]:
"""
    [+] SparkConf 객체 생성 및 설정
    - setMaster(), 마스터 노드 위치: "local"
    - 앱이름: "transformations_actions"
"""

conf = SparkConf()\
    .setMaster('local')\
    .setAppName('rdd-transformations-and-actions')

In [3]:
# [+] SparkContext 객체 생성
sc = SparkContext(conf=conf)

In [4]:
# Spark 전체 설정 값 확인
sc.getConf().getAll()

[('spark.driver.port', '52232'),
 ('spark.rdd.compress', 'True'),
 ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'),
 ('spark.master', 'local'),
 ('spark.app.id', 'local-1790828219597'),
 ('spark.driver.host', 'DESKTOP-MOEPBJT'),
 ('spark.sql.artifact.isolation.enabled', 'false'),
 ('spark.app.startTime', '1790828215523'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch

In [5]:
# SpackContext 객체 하나 더 생성하기(과연?)
sc = SparkContext(conf=conf)

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=rdd-transformations-and-actions, master=local) created by __init__ at C:\Users\chach\AppData\Local\Temp\ipykernel_8588\824348036.py:2 

In [6]:
# SparkContext는 프로그램 내에서 하나의 객체만 생성할 수 있다. (= Singleton 패턴)
# SparkContext 객체 해제하기
sc.stop()

In [7]:
# 다시 생성하기
sc = SparkContext(conf=conf)

### **기본적인 액션 메서드들**
+ ```collect()```: 모든 파티션의 데이터들을 드라이버 노드로 수집 및 리스트로 반환
+ ```countByValue()```: 해당 값의 발생 빈도 계산
+ ```take(n)```: 처음 ```n```개의 값을 반환
+ ```first()```: 첫 번째 값을 반환
+ ```count()```: 값의 갯수를 계산

In [13]:
"""
    [+] List로부터 RDD 객체 생성
    - SparkContext.parallelize()
    - 리스트 값: 
    "짜장면", "마라탕", "짬뽕", 
    "떡볶이", "쌀국수", "짬뽕", 
    "짜장면", "짜장면", "짜장면", 
    "라면", "우동", "라면"
"""

food_lst = [
    "짜장면", "마라탕", "짬뽕", 
    "떡볶이", "쌀국수", "짬뽕", 
    "짜장면", "짜장면", "짜장면", 
    "라면", "우동", "라면"]
       
# [+] lst로부터 RDD 객체 생성하기
foods = sc.parallelize(food_lst)

In [14]:
# [+] foods RDD 내용 출력
foods.collect()

['짜장면', '마라탕', '짬뽕', '떡볶이', '쌀국수', '짬뽕', '짜장면', '짜장면', '짜장면', '라면', '우동', '라면']

In [15]:
# [+] RDD 원소 값 별로 발생 빈도 계산
foods.countByValue()

defaultdict(int,
            {'짜장면': 4,
             '마라탕': 1,
             '짬뽕': 2,
             '떡볶이': 1,
             '쌀국수': 1,
             '라면': 2,
             '우동': 1})

In [16]:
# [+] take(n): 처음 n개의 값을 출력
foods.take(5)

['짜장면', '마라탕', '짬뽕', '떡볶이', '쌀국수']

In [17]:
# [+] first(): 첫 번째 값을 출력
foods.first()

'짜장면'

In [ ]:
# count(): 값 갯수 출력(중복된 원소 포함)
foods.count()

In [ ]:
# distinct(): unique한 원소로 구성된 RDD를 생성하는 Transformation

unique_foods = foods.distinct()
unique_foods.collect()

In [11]:
from pyspark import SparkConf, SparkContext

conf = (
    SparkConf()
    .setMaster("local[*]")
    .setAppName("uber-date-trips")
    .set("spark.driver.host", "127.0.0.1")
    .set("spark.driver.bindAddress", "127.0.0.1")
)

sc = SparkContext(conf=conf)

print(sc.version)

4.2.0


### **기본적인 변환 메서드들**

In [18]:
# Transformations 은 지연 실행된다.
sc.parallelize([1, 2, 3]).map(lambda x: x + 2)

PythonRDD[6] at RDD at PythonRDD.scala:59

In [19]:
# Actions 를 만나면 즉시 실행된다.
sc.parallelize([1, 2, 3]).map(lambda x: x + 2).collect()

[3, 4, 5]

In [20]:
sc.parallelize([1, 2, 3]).map(lambda x: x * 2).collect()

[2, 4, 6]

In [21]:
movies = [
    "그린 북",
    "매트릭스",
    "토이 스토리",
    "캐스트 어웨이",
    "포드 V 페라리",
    "보헤미안 랩소디",
    "빽 투 더 퓨처",
    "반지의 제왕",
    "죽은 시인의 사회"
]

In [22]:
# [+] 리스트(movies) -> RDD 객체(moviesRDD) 생성
moviesRDD = sc.parallelize(movies)
moviesRDD.collect()

['그린 북',
 '매트릭스',
 '토이 스토리',
 '캐스트 어웨이',
 '포드 V 페라리',
 '보헤미안 랩소디',
 '빽 투 더 퓨처',
 '반지의 제왕',
 '죽은 시인의 사회']

In [23]:
# [+] flatMap을 이용한 Tokenizing
flatMovies = moviesRDD.flatMap(lambda x: x.split(" "))
flatMovies.collect()

['그린',
 '북',
 '매트릭스',
 '토이',
 '스토리',
 '캐스트',
 '어웨이',
 '포드',
 'V',
 '페라리',
 '보헤미안',
 '랩소디',
 '빽',
 '투',
 '더',
 '퓨처',
 '반지의',
 '제왕',
 '죽은',
 '시인의',
 '사회']

In [24]:
"""
    [+] flatMovies 에서 '매트릭스'를 제외
    - filter(lambda ...)
    - 저장할 객체: filteredMovies
"""

filteredMovies = flatMovies.filter(lambda x : x != '매트릭스')

In [25]:
filteredMovies.collect()

['그린',
 '북',
 '토이',
 '스토리',
 '캐스트',
 '어웨이',
 '포드',
 'V',
 '페라리',
 '보헤미안',
 '랩소디',
 '빽',
 '투',
 '더',
 '퓨처',
 '반지의',
 '제왕',
 '죽은',
 '시인의',
 '사회']

### **집합 연산 관련 변환 메서드들**
+ ```intersection(rdd)```: 교집합 계산
+ ```union(rdd)```: 합집합 계산
+ ```subtract(rdd)```: 차집합 계산

In [32]:
# [+] 두 개의 정수 리스트로부터 RDD 객체 생성(num1, num2)
num1 = sc.parallelize([1,2,3,4,3])
num2 = sc.parallelize([1,2,3,5,4])

In [33]:
# [+] 교집합 구하기
num1.intersection(num2).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 15.0 failed 1 times, most recent failure: Lost task 0.0 in stage 15.0 (TID 96) (DESKTOP-MOEPBJT executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:704)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:686)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1068)
	at org.apache.spark.api.python.PythonRunner$$anon$3.

In [34]:
# [+] 합집합 구하기
num1.union(num2).collect()

[1, 2, 3, 4, 3, 1, 2, 3, 5, 4]

In [37]:
# [+] 차집합 구하기
num1.subtract(num2).collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 20.0 failed 1 times, most recent failure: Lost task 1.0 in stage 20.0 (TID 131) (DESKTOP-MOEPBJT executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:704)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:686)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1068)
	at org.apache.spark.api.python.PythonRunner$$anon$3

### **랜덤 샘플링 함수**
+ ```sample(withReplacement, fraction)```: 원소를 랜덤 샘플링하여 새로운 RDD를 생성하는 narrow transformation
    + ```withReplacement```, bool[optional]: 중복 샘플링 허용 여부
    + ```fraction```, float[optional]: 샘플링 확률[0~1], 확률이므로 정확한 사이즈가 보장되지 않음
    + ```seed```: 결과 재현(reproducibility)을 위한 설정 값

In [38]:
# [+] 합집합 생성(num1 + num2 -> numUnion)
numUnion = num1.union(num2)
numUnion.collect()

[1, 2, 3, 4, 3, 1, 2, 3, 5, 4]

In [40]:
# [+] 샘플링: 중복 허용, 50%
numUnion.sample(True, .5).collect()

[1, 1, 3, 2]

In [41]:
# [+] seed 파라미터를 설정하면 동일한 결과를 반복적으로 얻을 수 있다.
numUnion.sample(True, .5, seed=5)

PythonRDD[70] at RDD at PythonRDD.scala:59

### **그룹핑 연산**
+ ```groupBy()```: 특정 기준(함수로 정의)으로 그룹핑을 수행하는 wide transformation

In [ ]:
students = sc.parallelize([
    "이수경","이학선","이준솔",
    "이채원","윤지원","김태근",
    "서지유","이다영","정승윤",
    "박민규","이예소","이영림",
    "정승준","이용희","유형원",
    "박병규","방유리","박진영",
    "장민혁","김도훈"
])

In [ ]:
# [+] 원소의 첫 번째 값을 기준으로 그룹핑
studentGroup = students.groupBy(lambda x : x[0])
studentGroup.collect()

In [ ]:
# collect() 결과를 리스트 객체(res)에 저장
res = studentGroup.collect()
res

In [ ]:
res[0][1]

In [ ]:
# ResultIterable 객체는 list로 쉽게 변환 가능하다.
list(res[0][1])

In [ ]:
# [+] for 문을 이용하여 구조적으로 결과를 출력
for (k, v) in res:
    print(k, list(v))

In [ ]:
# groupBy()는 모드 연산과 같이 수학 연산에도 사용될 수 있다.
nums = sc.parallelize([1,2,3,4,5,6,7,8,9,10])

In [ ]:
# 홀수/짝수로 그룹핑
nums.groupBy(lambda x: x % 2).collect()

In [ ]:
res = nums.groupBy(lambda x: x % 2).collect()

In [ ]:
# [+] '1' 그룹에 해당되는 원소 리스트 출력하기
list(res[0][1])

In [ ]:
# [+] '0' 그룹에 해당되는 원소 리스트 출력하기
list(res[1][1])